# Resumable track record: pilot


**The pilot (instruction 64.25).** The production hold-out computation (`track_record.evaluate` + set hold-outs, exactly as `build_track_record.py`) for 3 methods on the Toxoplasma label `screen_egress_phenotype`, cut into one partition per method. Each partition carries a fingerprint of the node table bytes, the method settings for this label and the code that computes the calls.

**Pre-declared steps:** (1) run with a budget of one partition, so it stops after the first; (2) resume: only the other two are built; (3) replay: nothing is built; (4) change one setting of one method: only that partition rebuilds; (5) compare with the shipped track record.

Built 2026-11-07.

In [1]:
import os, json, shutil, dataclasses, pandas as pd
from parascape import precompute_track_record as P, track_record as T
work = os.path.join(OUT, 'run'); shutil.rmtree(work, ignore_errors=True)
parts = P.plan('Tg', [TARGET], METHODS)
pd.DataFrame([dict(method=p.strategy, settings=p.settings, fingerprint=p.fingerprint[:16]) for p in parts])

,method,settings,fingerprint
0,feature_knn,"{""k"": 15, ""min_share"": 0.3, ""target"": ""screen_egress_phenotype""}",88335b6d1e6e20ee
1,map_neighbours,"{""k"": 15, ""min_dist"": 0.1, ""min_share"": 0.3, ""n_neighbors"": 25, ""sample"": 4000, ""target"": ""screen_egress_phenotype""}",8ab05384bce597f5
2,layer_vote,"{""k"": 15, ""target"": ""screen_egress_phenotype""}",8fb26e5ef89a06a6


## 1-3 · Stop, resume, replay

In [2]:
first = P.run(parts, work, P.Budget(max_jobs=1, max_seconds=3600, memory_limit_bytes=12 << 30))
second = P.run(parts, work, P.Budget(max_seconds=3600, memory_limit_bytes=12 << 30))
third = P.run(parts, work)
steps = pd.DataFrame({'stop after one': first, 'resume': second, 'replay': third})
assert list(first.values()) == ['built', 'pending', 'pending']
assert list(second.values()) == ['reused', 'built', 'built'] and set(third.values()) == {'reused'}
steps

,stop after one,resume,replay
Tg__screen_egress_phenotype__feature_knn__88335b6d1e6e20ee,built,reused,reused
Tg__screen_egress_phenotype__map_neighbours__8ab05384bce597f5,pending,built,reused
Tg__screen_egress_phenotype__layer_vote__8fb26e5ef89a06a6,pending,built,reused


## 4 · One changed setting rebuilds one partition

The `layer_vote` partition is given a different settings record (as if its default changed); the other two keep their fingerprints and are reused.

In [3]:
changed = [dataclasses.replace(p, settings=p.settings[:-1] + ', "pilot_change": 1}') if p.strategy == 'layer_vote' else p for p in parts]
built = []
def recording(p):
    built.append(p.strategy)
    return P.build_partition(dataclasses.replace(p, settings=[q for q in parts if q.strategy == p.strategy][0].settings))
fourth = P.run(changed, work, builder=recording)
assert built == ['layer_vote'], built
fourth

{'Tg__screen_egress_phenotype__feature_knn__88335b6d1e6e20ee': 'reused', 'Tg__screen_egress_phenotype__map_neighbours__8ab05384bce597f5': 'reused', 'Tg__screen_egress_phenotype__layer_vote__05535630e714f418': 'built'}

## 5 · Cost per partition, and agreement with the shipped record

Peak memory is the process high-water mark (VmHWM) after each partition, so it includes earlier work in the same process; it is not an isolated per-partition peak.

In [4]:
cost = P.report(work)
cost['peak_rss_MiB'] = (cost.peak_rss_bytes / 2**20).round(1); cost['KiB'] = (cost.bytes / 1024).round(1)
table = P.assemble(work, parts)
shipped = T.shipped('Tg')
rows = []
for m in METHODS:
    a = table[(table.strategy.astype(str) == m) & (table['mode'].astype(str) == 'together')]
    s = shipped[(shipped.strategy.astype(str) == m) & (shipped.target.astype(str) == TARGET) & (shipped['mode'].astype(str) == 'together')]
    s = s[s.setting_key.astype(str) == s.setting_key.astype(str).iloc[0]] if len(s) else s
    j = a.merge(s, on='gene_id', suffixes=('', '_shipped'))
    both_missing = j.prediction.isna() & j.prediction_shipped.isna()  # an abstention is a missing call on both sides
    same = ((j.prediction.astype(str) == j.prediction_shipped.astype(str)) | both_missing).mean() if len(j) else float('nan')
    rows.append(dict(method=m, genes_now=a.gene_id.nunique(), genes_shipped=s.gene_id.nunique(), both=len(j),
                     same_call=round(float(same), 3), accuracy_now=round(float(a.correct.fillna(False).mean()), 3),
                     accuracy_shipped=round(float(s.correct.fillna(False).mean()), 3) if len(s) else None))
agree = pd.DataFrame(rows)
cost.to_csv(os.path.join(OUT, 'cost.tsv'), sep='\t', index=False); agree.to_csv(os.path.join(OUT, 'agreement.tsv'), sep='\t', index=False)
storage = sum(os.path.getsize(os.path.join(dp, f)) for dp, _d, fs in os.walk(work) for f in fs)
json.dump(dict(target=TARGET, methods=METHODS, steps={k: dict(v) for k, v in dict(first=first, second=second, third=third, fourth=fourth).items()},
               storage_bytes=storage, total_seconds=float(cost.seconds.sum())), open(os.path.join(OUT, 'summary.json'), 'w'), indent=1)
print('storage', storage, 'bytes; total build seconds', round(float(cost.seconds.sum()), 1))
print(cost[['partition', 'status', 'rows', 'seconds', 'peak_rss_MiB', 'KiB']].to_string(index=False))
agree

storage 81246 bytes; total build seconds 60.9
                                                    partition status  rows  seconds  peak_rss_MiB  KiB
   Tg__screen_egress_phenotype__feature_knn__88335b6d1e6e20ee     ok  1140    2.475         389.1 21.7
    Tg__screen_egress_phenotype__layer_vote__05535630e714f418     ok  1140   17.905         679.6 15.9
    Tg__screen_egress_phenotype__layer_vote__8fb26e5ef89a06a6     ok  1140   19.090         679.6 15.9
Tg__screen_egress_phenotype__map_neighbours__8ab05384bce597f5     ok  1140   21.415         679.6 21.7


,method,genes_now,genes_shipped,both,same_call,accuracy_now,accuracy_shipped
0,feature_knn,318,318,318,1,0.89,0.89
1,map_neighbours,318,318,318,1,0.887,0.887
2,layer_vote,318,318,318,1,0.547,0.547
